# 🏢 Business Entity Resolution — Kaggle Training & Inference Pipeline
**ML Challenge 2026 | Macro F0.5 Metric Optimization**

This notebook is customized to run **end-to-end on Kaggle** (as well as AWS SageMaker and local environments):
- 🚀 **Zero-Configuration Setup**: Automatically clones `https://github.com/caviraj/ml_challege.git` if opened standalone on Kaggle.
- 🔍 **Smart Dataset Auto-Discovery**: Automatically locates parquet files anywhere in `/kaggle/input/` or local `data/` folders.
- ⚡ **LightGBM Matching Model**: 32 pairwise similarity features + 5-Fold GroupKFold threshold tuning for Macro F0.5.
- 🧠 **Memory-Efficient Inference**: Country-partitioned candidate generation and streaming TSV prediction (stays well within Kaggle's 30 GB RAM).
- 🛡️ **Submission Validation**: Automatic format, completeness, and prefix validation matching competition rules.
- 📦 **1-Click Download**: Generates `/kaggle/working/submission.zip` directly accessible in the Kaggle Output viewer.

---
### ⚙️ Quick Start on Kaggle:
1. In the right sidebar under **Settings** -> **Internet**, toggle Internet to **ON** (required to install packages and clone repo).
2. (Optional) If parquet files are not in the repo, click **+ Add Data** (top right) and attach your dataset.
3. Click **Run All** (or run cells one by one from top to bottom).
4. Download `submission.zip` from the right-hand **Output** tab.

## 1. Environment Setup & Dependency Installation

In [ ]:
import os
import sys
import glob
import shutil
import zipfile

# 1. Detect environment (Kaggle vs Local / SageMaker)
IS_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ or os.path.exists('/kaggle/working')

print(f"Environment check -> Running on Kaggle: {IS_KAGGLE}")

if IS_KAGGLE:
    WORK_DIR = "/kaggle/working"
    REPO_DIR = os.path.join(WORK_DIR, "ml_challenge")
    PROJECT_DIR = os.path.join(REPO_DIR, "business_entity_resolution")
    
    # Clone repository if running standalone
    if not os.path.exists(REPO_DIR):
        print("Cloning repository https://github.com/caviraj/ml_challege.git...")
        !git clone https://github.com/caviraj/ml_challege.git {REPO_DIR}
    else:
        print(f"Repository already present at {REPO_DIR}. Pulling latest updates...")
        !cd {REPO_DIR} && git pull origin main
    
    # Navigate to project root
    if os.path.exists(PROJECT_DIR):
        os.chdir(PROJECT_DIR)
    elif os.path.exists(REPO_DIR):
        os.chdir(REPO_DIR)
else:
    # Local / SageMaker environment
    if os.path.basename(os.getcwd()) == 'notebooks':
        os.chdir('..')

# Ensure python finds src modules
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print(f"Active project root: {os.getcwd()}")

In [ ]:
# 2. Install dependencies (RapidFuzz, Polars, LightGBM, PyArrow, etc.)
!pip install -r requirements.txt --quiet
print("All required libraries installed successfully!")

## 📁 2. Dataset Auto-Discovery & Verification
The cell below automatically scans `/kaggle/input` (and local directories) to locate the processed parquet files.
If you attached your dataset via Kaggle's **+ Add Data**, it will locate the folder instantly without needing manual path changes.

In [ ]:
def locate_dataset():
    """
    Scans common Kaggle and local paths to find train_source1.parquet.
    Returns the folder path containing the parquet files.
    """
    required_file = "train_source1.parquet"
    search_roots = ["/kaggle/input", "data/processed", "data", ".", "..", "/kaggle/working"]
    
    for root in search_roots:
        if not os.path.exists(root):
            continue
        matches = glob.glob(f"{root}/**/{required_file}", recursive=True)
        if matches:
            found_dir = os.path.dirname(os.path.abspath(matches[0]))
            return found_dir
            
    return None

data_dir = locate_dataset()
if data_dir is None:
    data_dir = os.path.abspath("data/processed")

print(f"Dataset location detected at: {data_dir}\n")

files = [
    "train_source1.parquet", "train_source2.parquet", "train_source3.parquet", "train_ground_truth.parquet",
    "test_source1.parquet", "test_source2.parquet", "test_source3.parquet"
]

missing = []
for f in files:
    f_path = os.path.join(data_dir, f)
    if os.path.exists(f_path):
        size_mb = os.path.getsize(f_path) / (1024 * 1024)
        print(f"  [OK] {f} ({size_mb:.1f} MB)")
    else:
        print(f"  [MISSING] {f}")
        missing.append(f)

if missing:
    print(f"\n[NOTE] {len(missing)} file(s) missing in {data_dir}.")
    print("If running on Kaggle, upload your processed parquet files via '+ Add Data' -> 'Upload a Dataset'.")
    print("The notebook will automatically locate them under /kaggle/input/.")

## 🎯 3. Train LightGBM Matching Model
- Generates candidate pairs for training samples across Source 1, 2, and 3.
- Extracts 32 pairwise similarity features (token overlap, fuzzy ratios, Levenshtein, country/postal match, etc.).
- Evaluates 5-fold GroupKFold to find the optimal decision threshold that maximizes **Macro F0.5**.
- **Tip**: Set `SAMPLE_SIZE = 150000` for balanced training in ~10-15 minutes, or `SAMPLE_SIZE = 0` to train on 100% of data.

In [ ]:
os.makedirs("models", exist_ok=True)

# Training sample size:
# 150000 = Balanced training (~10-15 mins on Kaggle CPU)
# 0 = Train on entire dataset
SAMPLE_SIZE = 150000

s1_train = os.path.join(data_dir, "train_source1.parquet")
s2_train = os.path.join(data_dir, "train_source2.parquet")
s3_train = os.path.join(data_dir, "train_source3.parquet")
gt_train = os.path.join(data_dir, "train_ground_truth.parquet")
model_output = "models/lgb_model.joblib"

print(f"Launching training pipeline (Sample Size: {SAMPLE_SIZE})...")
!python -m src.modeling.train \
    --s1-path "{s1_train}" \
    --s2-path "{s2_train}" \
    --s3-path "{s3_train}" \
    --gt-path "{gt_train}" \
    --sample-size {SAMPLE_SIZE} \
    --output-model "{model_output}"

print(f"\nTrained model artifact successfully saved to: {model_output}")

## ⚡ 4. Test Inference (Country-Partitioned, RAM-Safe)
- Evaluates all **1,732,544 test Source 1 records** against indexed Source 2 and 3 records.
- Processes country-by-country partitions so RAM usage remains under 8 GB (well within Kaggle's 30 GB limit).
- Streams predictions into `matching_results.tsv` and `candidate_pairs.tsv`.

In [ ]:
# Output directory: on Kaggle, write to /kaggle/working/output for direct access
output_dir = "/kaggle/working/output" if IS_KAGGLE else "output"
os.makedirs(output_dir, exist_ok=True)

s1_test = os.path.join(data_dir, "test_source1.parquet")
s2_test = os.path.join(data_dir, "test_source2.parquet")
s3_test = os.path.join(data_dir, "test_source3.parquet")

print("Launching inference pipeline on test dataset...")
!python -m src.modeling.predict \
    --model-path "{model_output}" \
    --test-s1 "{s1_test}" \
    --test-s2 "{s2_test}" \
    --test-s3 "{s3_test}" \
    --output-dir "{output_dir}"

print(f"\nInference completed! Output TSV files generated in: {output_dir}")

## ✅ 5. Validate Submission Files
Strictly validates generated TSVs against all ML Challenge competition rules:
1. Exact headers: `source1_entity_id\tmatched_entity_ids` & `source1_entity_id\tcandidate_entity_ids`.
2. Exact row count: **1,732,544** rows matching `test_source1`.
3. Entity ID prefixes: only `s2_` or `s3_` allowed (no self matches `s1_`).
4. Tab-delimited formatting without trailing whitespace or malformed rows.

In [ ]:
matching_tsv = os.path.join(output_dir, "matching_results.tsv")
candidate_tsv = os.path.join(output_dir, "candidate_pairs.tsv")

print("Executing submission verification...")
!python scripts/validate_submission.py \
    --matching "{matching_tsv}" \
    --candidate "{candidate_tsv}" \
    --test-s1 "{s1_test}"

## 📦 6. Package Submission ZIP & Download
Compresses `matching_results.tsv` and `candidate_pairs.tsv` into `submission.zip`.
On Kaggle, saving to `/kaggle/working/submission.zip` exposes the file directly in the right-hand **Output** pane for 1-click download.

In [ ]:
if IS_KAGGLE:
    zip_dest = "/kaggle/working/submission.zip"
else:
    zip_dest = os.path.join(output_dir, "submission.zip")

print(f"Creating submission zip archive at: {zip_dest}...")
with zipfile.ZipFile(zip_dest, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    z.write(matching_tsv, arcname="matching_results.tsv")
    z.write(candidate_tsv, arcname="candidate_pairs.tsv")

size_mb = os.path.getsize(zip_dest) / (1024 * 1024)
print(f"\n🎉 SUCCESS! Submission archive ready: {zip_dest}")
print(f"Archive Size: {size_mb:.2f} MB")

if IS_KAGGLE:
    print("\n" + "="*50)
    print("📥 HOW TO DOWNLOAD YOUR SUBMISSION FILE FROM KAGGLE:")
    print("="*50)
    print("1. Look at the right sidebar of your Kaggle notebook window.")
    print("2. Expand the 'Output' section under '/kaggle/working'.")
    print("3. Locate 'submission.zip'.")
    print("4. Click the three dots (...) next to it and select 'Download'!")
    print("="*50)
else:
    print(f"\nReady to download from your Jupyter file browser at: {zip_dest}")